In [2]:
import os 
import json
import pandas as pd #to keep track of the questions and answers
import traceback #to keep track of the errors. 

In [3]:
from langchain_community.llms import HuggingFaceHub

c:\Users\admin\mcqgen\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from dotenv import load_dotenv, dotenv_values
from pathlib import Path
import os

env_path = Path(r"C:\Users\admin\mcqgen\.env")
print(dotenv_values(env_path).keys())  # should show HF_API_KEY

load_dotenv(env_path, override=True)
KEY = os.environ["HF_API_KEY"]
print(KEY is None)  # should be False

odict_keys(['HF_API_KEY'])
False


In [194]:
KEY=os.getenv("HF_API_KEY")


In [154]:
import os
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"  # small; pick what fits your RAM/GPU
tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    return_full_text=False,
    max_new_tokens=2000,
    # do NOT pass max_length=20 anywhere
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights: 100%|██████████| 338/338 [00:03<00:00, 91.84it/s] 
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [155]:
llm

HuggingFacePipeline(pipeline=TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen2.5-1.5B-Instruct')

In [156]:
import sys
print(sys.executable)

c:\Users\admin\mcqgen\env\python.exe


In [157]:
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_community.llms import HuggingFaceHub
from pypdf import PdfReader

In [158]:
#This response is in form of dictionary its needs to be connverted in json serializer later
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}
#for correct aswer we meed another template

In [159]:
TEMPLATE = """
Text: {text}

You are an expert MCQ maker. Given the above text, it is your job to create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide.
Ensure to make {number} MCQs

### RESPONSE_JSON
Return ONLY the JSON. Do not repeat the input text or any extra words.
{response_json}
""".strip()

In [160]:
#design my imput prompt amd output prompt for llm using llm chain for mcq generation
#prompt template
from langchain_classic.prompts import PromptTemplate

quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE,
)


In [161]:
#CREATE LLM CHAIN TO CONNECT COMPOENTS 
#IT WILL CONNECT LLM WITH PROMPT. 
quiz_chain = LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)

In [162]:
TEMPLATE2 = """
You are an expert English grammarian and writer.

Given a Multiple Choice Quiz for {subject} students, you must:
- Analyze the quiz complexity in <= 50 words.
- If needed, rewrite ONLY the questions/options to better fit the student level.
- Keep the SAME number of MCQs and the SAME numbering keys as the input (e.g. "1","2",...).
- Ensure exactly one correct option per MCQ.

Quiz_MCQs (JSON):
{quiz}

Return ONLY valid JSON (no markdown, no extra text) in this exact schema:
{{
  "complexity_analysis": "... <= 50 words ...",
  "updated_quiz": {{
    "1": {{
      "mcq": "...",
      "options": {{"a":"...","b":"...","c":"...","d":"..."}},
      "correct": "a|b|c|d"
    }}
  }}
}}
""".strip()

In [163]:
from langchain_classic.prompts import PromptTemplate

#in the second input it comes from the user side

quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2,
)

In [164]:
#the second chain will evaluate the quiz, the output will be collected in the review variable
from langchain_classic.chains import LLMChain

review_chain = LLMChain(
    llm=llm,
    prompt=quiz_evaluation_prompt,
    verbose=True,
    output_key="review",
)

In [165]:
#not connect both chains using simple sequential chain and created an object of sequential chain
generate_evaluate_chain = SequentialChain(
    chains=[quiz_chain, review_chain],
    input_variables=["text", "number", "subject", "tone", "response_json"],
    output_variables=["quiz", "review"],
    verbose=True,
)

In [178]:
#r_ means read it.
file_path=r"C:\Users\admin\mcqgen\data.txt"

In [179]:
file_path

'C:\\Users\\admin\\mcqgen\\data.txt'

In [168]:
#now you need a data. create a data.txt file and add the text you want to generate the quiz from.
#use thois code to read the data.txt file
with open(file_path, "r") as file:
    text = file.read()


In [120]:
print(text)

The field of materials informatics1,2 suffers from lack of data readiness and
data accessibility. Although materials data can be systematically generated
through computational and physical experiments, a substantial amount of
historical data is trapped in published literature.An ever-growing volume of
data is continually released in scientific journal articles, but this data frequently
exists in unstructured natural language text formats, posing challenges
for immediate utilization by modern informatics that rely on the
availability of structured datasets. Natural language processing (NLP)
techniques implemented in materials science seek to automatically extract
materials insights,materials properties, and synthesis data from a corpus of
text documents, and propose hypotheses and designs for new materials3–5.
After acquiring the corpus, a series of complex NLP operations are performed
which include turning texts into smaller units called tokens,
recognizing key entities (such as materi

In [170]:
#serialize the python dictionary into json format
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [180]:
#THE VARIABLES YOU DEFINED EARLIER NEED TO BE CLARIFIEDN
number=5
subject="machine learning"
tone="simple"	



In [181]:
#The final step for response. using openAi get_openai_callbaimport json

response = generate_evaluate_chain.invoke(
    {
        "text": text,
        "number": number,
        "subject": subject,
        "tone": tone,
        "response_json": json.dumps(RESPONSE_JSON),
    }
)



Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Text: The field of materials informatics1,2 suffers from lack of data readiness and
data accessibility. Although materials data can be systematically generated
through computational and physical experiments, a substantial amount of
historical data is trapped in published literature.An ever-growing volume of
data is continually released in scientific journal articles, but this data frequently
exists in unstructured natural language text formats, posing challenges
for immediate utilization by modern informatics that rely on the
availability of structured datasets. Natural language processing (NLP)
techniques implemented in materials science seek to automatically extract
materials insights,materials properties, and synthesis data from a corpus of
text documents, and propose hypotheses and designs for new materials3–5.
After acquiring the corpus, a series of complex NLP operations are perf

Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
You are an expert English grammarian and writer.

Given a Multiple Choice Quiz for machine learning students, you must:
- Analyze the quiz complexity in <= 50 words.
- If needed, rewrite ONLY the questions/options to better fit the student level.
- Keep the SAME number of MCQs and the SAME numbering keys as the input (e.g. "1","2",...).
- Ensure exactly one correct option per MCQ.

Quiz_MCQs (JSON):
 

## RESPONSE_JSON
```json
{
    "1": {
        "mcq": "What tool was used to fine-tune the LLaMa-2 model?",
        "options": {
            "a": "Codex",
            "b": "ChatGPT",
            "c": "Code LLaMa",
            "d": "GPT-3.5"
        },
        "correct": "d"
    },
    "2": {
        "mcq": "Which of the following tools were used for data extraction?",
        "options": {
            "a": "MaterialsBERT",
            "b": "LlaMa",
            "c": "GPT-3.5",
            "d": "All of the above"


In [182]:
quiz=response.get("quiz")

In [183]:
s = extract_json_object(quiz)
print(repr(s[:400]))

'{\n    "1": {\n        "mcq": "What tool was used to fine-tune the LLaMa-2 model?",\n        "options": {\n            "a": "Codex",\n            "b": "ChatGPT",\n            "c": "Code LLaMa",\n            "d": "GPT-3.5"\n        },\n        "correct": "d"\n    },\n    "2": {\n        "mcq": "Which of the following tools were used for data extraction?",\n        "options": {\n            "a": "MaterialsBERT",\n'


In [184]:
import json

def parse_first_json_object(s: str):
    s = s.strip()
    start = s.find("{")
    if start == -1:
        raise ValueError("No JSON object found in output.")
    obj, _ = json.JSONDecoder().raw_decode(s[start:])
    return obj

quiz_data = parse_first_json_object(quiz)

In [185]:
print(quiz)

 

## RESPONSE_JSON
```json
{
    "1": {
        "mcq": "What tool was used to fine-tune the LLaMa-2 model?",
        "options": {
            "a": "Codex",
            "b": "ChatGPT",
            "c": "Code LLaMa",
            "d": "GPT-3.5"
        },
        "correct": "d"
    },
    "2": {
        "mcq": "Which of the following tools were used for data extraction?",
        "options": {
            "a": "MaterialsBERT",
            "b": "LlaMa",
            "c": "GPT-3.5",
            "d": "All of the above"
        },
        "correct": "d"
    },
    "3": {
        "mcq": "How many paragraphs were processed in total?",
        "options": {
            "a": "1000",
            "b": "233000",
            "c": "6179",
            "d": "7169"
        },
        "correct": "c"
    },
    "4": {
        "mcq": "What metric was used to compare the performance of the models?",
        "options": {
            "a": "Accuracy",
            "b": "Time",
            "c": "Cost",
            

In [186]:
#WE NEED TO CREATE A data frame for which we will create a list first 
quiz_table_data = []

for key, value in quiz_data.items():
    mcq = value["mcq"]
    options = " | ".join([f"{opt}: {opt_val}" for opt, opt_val in value["options"].items()])
    correct = value["correct"]

    quiz_table_data.append(
        {"MCQ": mcq, "Choices": options, "Correct": correct}
    )

In [187]:
quiz_table_data


[{'MCQ': 'What tool was used to fine-tune the LLaMa-2 model?',
  'Choices': 'a: Codex | b: ChatGPT | c: Code LLaMa | d: GPT-3.5',
  'Correct': 'd'},
 {'MCQ': 'Which of the following tools were used for data extraction?',
  'Choices': 'a: MaterialsBERT | b: LlaMa | c: GPT-3.5 | d: All of the above',
  'Correct': 'd'},
 {'MCQ': 'How many paragraphs were processed in total?',
  'Choices': 'a: 1000 | b: 233000 | c: 6179 | d: 7169',
  'Correct': 'c'},
 {'MCQ': 'What metric was used to compare the performance of the models?',
  'Choices': 'a: Accuracy | b: Time | c: Cost | d: Quality',
  'Correct': 'c'},
 {'MCQ': 'Which of the following statements is true regarding the use of LLMs in data extraction?',
  'Choices': 'a: LLMs cannot be used for data extraction. | b: Using LLMs improves the quality of data extraction. | c: LLMs require a lot of manual effort. | d: LLMs can replace traditional NER models.',
  'Correct': 'b'}]

In [190]:
#convert into data frame
quiz=pd.DataFrame(quiz_table_data)

In [193]:
#convert it into a CSV file
quiz.to_csv("machine_learning_quiz.csv", index=False)